#### Input data processing and one-hot encoding of the simulated dataset

In [1]:
import pickle
import sys
import numpy as np
from ete3 import Tree
from sys import exit

## read the simulated multiple sequence alignment
with open("../data/processed/training/simulated/simulated_msa.pkl", 'rb') as file_handle:
    msa = pickle.load(file_handle)

## read the LG replacement matrix
with open("../data/processed/random/LG_matrix.pkl", 'rb') as file_handle:
    LG_matrix = pickle.load(file_handle)
amino_acids = LG_matrix['amino_acids']
aa2idx = {}
for i in range(len(amino_acids)):
    aa2idx[amino_acids[i]] = i

## convert msa into numbers from 0 to 19
len_protein = len(msa[list(msa.keys())[0]])
num_seq = len(msa.keys())

msa_keys = list(msa.keys())
msa_keys.sort()
msa_nume = np.zeros((num_seq, len_protein), dtype = np.int64)
for i in range(num_seq):
    msa_nume[i,:] = [aa2idx[s] for s in msa[msa_keys[i]]]

## convert msa into binary representation, i.e., using 0 or 1    
msa_binary = np.zeros((num_seq, len_protein, 20))
D = np.identity(20)
for i in range(num_seq):
    msa_binary[i,:,:] = D[msa_nume[i]]

## save processed msa
with open("../data/processed/training/simulated/msa_nume.pkl", 'wb') as file_handle:
    pickle.dump(msa_nume, file = file_handle)

with open("../data/processed/training/simulated/msa_binary.pkl", 'wb') as file_handle:
    pickle.dump(msa_binary, file = file_handle)

with open("../data/processed/training/simulated/msa_keys.pkl", 'wb') as file_handle:
    pickle.dump(msa_keys, file = file_handle)

#### the simulated sequences are splitted into two classes:
#### leaf node sequences and non-leaf node sequences.
#### only leaf node sequences are used for training latent
#### space models.

## save leaf node sequences
t = Tree("../data/processed/tree/simulated/random_tree.newick", format = 1)
t.name = str(len(t))
num_leaf = len(t)
msa_leaf_keys = [str(i) for i in range(num_leaf)]
msa_leaf_keys.sort()

msa_leaf_nume = np.zeros((num_leaf, len_protein), dtype = np.int64)
for i in range(num_leaf):
    msa_leaf_nume[i,:] = [aa2idx[s] for s in msa[msa_leaf_keys[i]]]

msa_leaf_binary = np.zeros((num_leaf, len_protein, 20))
D = np.identity(20)
for i in range(num_leaf):
    msa_leaf_binary[i,:,:] = D[msa_leaf_nume[i]]

with open("../data/processed/training/simulated/msa_leaf_nume.pkl", 'wb') as file_handle:
    pickle.dump(msa_leaf_nume, file = file_handle)

with open("../data/processed/training/simulated/msa_leaf_binary.pkl", 'wb') as file_handle:
    pickle.dump(msa_leaf_binary, file = file_handle)

with open("../data/processed/training/simulated/msa_leaf_keys.pkl", 'wb') as file_handle:
    pickle.dump(msa_leaf_keys, file = file_handle)


#### Input data processing and one-hot encoding of the cyclase dataset


In [ ]:
import pickle
import sys
import numpy as np
from sys import exit
from Bio import SeqIO
from Bio.Seq import Seq
from Bio.SeqRecord import SeqRecord

# User inputs
MSA_file = '../data/processed/fasta/cyclase/PF05147_hits_200_500aa_MSA.fasta'
query_seq_id = "WP_043991124.1/31-402" # NisC

# Define a list of amino acid characters
AA = ['R', 'H', 'K',
      'D', 'E',
      'S', 'T', 'N', 'Q',
      'C', 'G', 'P',
      'A', 'V', 'I', 'L', 'M', 'F', 'Y', 'W']

aa_to_index = {}
aa_to_index['-'] = 0
aa_to_index['.'] = 0
index_to_aa = {}
index_to_aa[0] = '-'
for idx, aa in enumerate(AA, start=1):
    aa_to_index[aa] = idx
    index_to_aa[idx] = aa

# Save aa_to_index
with open("../data/processed/training/cyclase/aa_to_index.pkl", 'wb') as file_handle:
    pickle.dump(aa_to_index, file_handle)

# Save index_to_aa
with open("../data/processed/training/cyclase/index_to_aa.pkl", 'wb') as file_handle:
    pickle.dump(index_to_aa, file_handle)

# Read all the sequences into a dictionary
def MSA_to_dict(MSA_file):
    seq_dict = {record.id: str(record.seq).upper() for record in SeqIO.parse(MSA_file, "fasta")}
    return seq_dict

# Remove a column if the query has a gap at that position
def remove_query_gap(seq_dict, query_seq_id):

    query_seq = seq_dict[query_seq_id] ## potentially with gaps
    idx = [ s == "-" or s == "." for s in query_seq]
    for k in seq_dict.keys():
        seq_dict[k] = [seq_dict[k][i] for i in range(len(seq_dict[k])) if idx[i] == False]
    query_seq = seq_dict[query_seq_id] ## without gaps

    return seq_dict, query_seq

# Remove gappy sequences
def remove_gappy_sequneces(seq_dict, query_seq_id):

    len_query_seq = len(query_seq)
    deleted_id_list = []
    deletion_reason = []
    for k in list(seq_dict.keys()):
        if seq_dict[k].count('X') > 0 or seq_dict[k].count('Z') > 0 or seq_dict[k].count('O') > 0:
            seq_dict.pop(k)
            deleted_id_list.append(k)
            deletion_reason.append('XZO')

        elif seq_dict[k].count("-") + seq_dict[k].count(".") > 0.2*len_query_seq:
            seq_dict.pop(k)
            deleted_id_list.append(k)
            deletion_reason.append('gappy')

    return seq_dict, deleted_id_list, deletion_reason

seq_dict = MSA_to_dict(MSA_file)
complete_query = seq_dict[query_seq_id]

seq_dict, query_seq = remove_query_gap(seq_dict, query_seq_id)
seq_dict, deleted_id_list, deletion_reason = remove_gappy_sequneces(seq_dict, query_seq_id)

# Save deleted_id_list
with open("../data/processed/training/cyclase/deleted_id_list.pkl", 'wb') as file_handle:
    pickle.dump(deleted_id_list, file_handle)

# Convert aa to index
id_list = []
seq_msa = []
for k in seq_dict.keys():
    id_list.append(k)
    seq_msa.append([aa_to_index[s] for s in seq_dict[k]])
seq_msa = np.array(seq_msa)

# Remove positions where too many sequences have gaps
pos_idx = []
for i in range(seq_msa.shape[1]):
    if np.sum(seq_msa[:,i] == 0) <= seq_msa.shape[0]*0.2:
        pos_idx.append(i)

with open("../data/processed/training/cyclase/seq_pos_idx.pkl", 'wb') as file_handle:
    pickle.dump(pos_idx, file_handle)

seq_msa = seq_msa[:, np.array(pos_idx)]

# Reweighting sequences
seq_weight = np.zeros(seq_msa.shape)
for j in range(seq_msa.shape[1]):
    aa_type, aa_counts = np.unique(seq_msa[:,j], return_counts = True)
    num_type = len(aa_type)
    aa_dict = {}
    for a in aa_type:
        aa_dict[a] = aa_counts[list(aa_type).index(a)]
    for i in range(seq_msa.shape[0]):
        seq_weight[i,j] = (1.0/num_type) * (1.0/aa_dict[seq_msa[i,j]])
tot_weight = np.sum(seq_weight)
seq_weight = seq_weight.sum(1) / tot_weight

# Save seq_weight
with open("../data/processed/training/cyclase/seq_weight.pkl", 'wb') as file_handle:
    pickle.dump(seq_weight, file_handle)

# Save seq_msa (one-hot encoded version)
with open("../data/processed/training/cyclase/seq_msa.pkl", 'wb') as file_handle:
    pickle.dump(seq_msa, file_handle)

# Decode idx back to aa
seq_msa_aa = []
for k in range(seq_msa.shape[0]):
    decoded_seq = "".join(index_to_aa[idx] for idx in seq_msa[k])
    seq_msa_aa.append(decoded_seq)

# Create a sequence dictionary with truncated sequences
seq_dict_truncated = {id: seq for id, seq in zip(id_list, seq_msa_aa)}

# Save seq_dict_truncated
with open("../data/processed/training/cyclase/seq_dict_truncated.pkl", 'wb') as file_handle:
     pickle.dump(seq_dict_truncated, file_handle)

# Create a dictionary that maps IDs to a sequence number
id2no = {id: f"seq{i+1}" for i, id in enumerate(seq_dict_truncated.keys())}

# Create a dictionary that maps IDs to a sequence number
no2id = {f"seq{i+1}": id for i, id in enumerate(seq_dict_truncated.keys())}

# Save the id2no dictionary to a pickle file
with open("../data/processed/training/cyclase/id2no.pkl", 'wb') as file_handle:
    pickle.dump(id2no, file_handle)

# Save the id2no dictionary to a pickle file
with open("../data/processed/training/cyclase/no2id.pkl", 'wb') as file_handle:
    pickle.dump(no2id, file_handle)

# Save seq_dict_truncated to a FASTA file
with open("../data/processed/fasta/cyclase/PF05147_hits_200_500aa_MSA_truncated.fasta", "w") as fasta_file:
    for id, seq in seq_dict_truncated.items():
        fasta_file.write(f">{id2no[id]}\n{seq}\n")

# Save the position number of the residues that are kept
template_pos_idx = []
for i, res in enumerate(complete_query):
    # If the sequence is empty, break the loop
    if not seq_dict_truncated[query_seq_id]:
        break
    if res == seq_dict_truncated[query_seq_id][0]:
        template_pos_idx.append(i+1)
        seq_dict_truncated[query_seq_id] = seq_dict_truncated[query_seq_id][1:]

# Save template_pos_idx
with open("../data/processed/training/cyclase/template_pos_idx.pkl", 'wb') as file_handle:
     pickle.dump(template_pos_idx, file_handle)

# Save keys_list
with open("../data/processed/training/cyclase/keys_list.pkl", 'wb') as file_handle:
    pickle.dump(id_list, file_handle)

# Save seq_dict_truncated
with open("../data/processed/training/cyclase/seq_dict_truncated.pkl", 'wb') as file_handle:
    pickle.dump(seq_dict_truncated, file_handle)

# Change aa numbering into binary
K = 21 ## num of classes of aa
D = np.identity(K)
num_seq = seq_msa.shape[0]
len_seq_msa = seq_msa.shape[1]
seq_msa_binary = np.zeros((num_seq, len_seq_msa, K))
for i in range(num_seq):
    seq_msa_binary[i,:,:] = D[seq_msa[i]]

# Save seq_msa_binary
with open("../data/processed/training/cyclase/seq_msa_binary.pkl", 'wb') as file_handle:
    pickle.dump(seq_msa_binary, file_handle)

#### Input data processing and one-hot encoding of the FDMO dataset

In [ ]:
import pickle
import sys
import numpy as np
from sys import exit
from Bio.Seq import Seq
from Bio import SeqIO
from Bio.SeqRecord import SeqRecord

file_name = "../data/processed/fasta/FDMO/PF01494_MSA.fasta"
query_seq_id = "B8M9J8" #TropB

# Read all the sequences into a dictionary
id_list = []
seq_list = []
seq_dict = {}
for record in SeqIO.parse(file_name, "fasta"):
    ID = record.id
    id_list.append(ID)
    seq_list.append(str(record.seq.upper()))
    seq_dict[ID] = str(record.seq.upper())

FMO = []
for idx, (seq, ID) in enumerate(zip(seq_list, id_list)):
    FMO.append(SeqRecord(Seq(seq), id=ID, description="")) 
with open("../data/processed/fasta/FDMO/PF01494_MSA_no_des.fasta", "w") as handle:
    SeqIO.write(FMO, handle, "fasta")
    
# Remove gaps in the query sequences
query_seq = seq_dict[query_seq_id] ## with gaps
idx = [ s == "-" or s == "." for s in query_seq]
for k in seq_dict.keys():
    seq_dict[k] = [seq_dict[k][i] for i in range(len(seq_dict[k])) if idx[i] == False]
query_seq = seq_dict[query_seq_id] ## without gaps

# Remove sequences with too many gaps
len_query_seq = len(query_seq)
seq_id = list(seq_dict.keys())
num_gaps = []
for k in seq_id:
    num_gaps.append(seq_dict[k].count("-") + seq_dict[k].count("."))
    if seq_dict[k].count("-") + seq_dict[k].count(".") > 0.2*len_query_seq:
        seq_dict.pop(k)

with open("../data/processed/training/FDMO/seq_dict.pkl", 'wb') as file_handle:
     pickle.dump(seq_dict, file_handle)

# Convert aa type into num 0-20
aa = ['R', 'H', 'K',
      'D', 'E',
      'S', 'T', 'N', 'Q',
      'C', 'G', 'P',
      'A', 'V', 'I', 'L', 'M', 'F', 'Y', 'W']
aa_index = {}
aa_index['-'] = 0
aa_index['.'] = 0
i = 1
for a in aa:
    aa_index[a] = i
    i += 1
with open("../data/processed/training/FDMO/aa_index.pkl", 'wb') as file_handle:
    pickle.dump(aa_index, file_handle)
    
seq_msa = []
keys_list = []
for k in seq_dict.keys():
    if seq_dict[k].count('X') > 0 or seq_dict[k].count('Z') or seq_dict[k].count('O')> 0:
        continue    
    seq_msa.append([aa_index[s] for s in seq_dict[k]])
    keys_list.append(k)    
seq_msa = np.array(seq_msa)

training_keys_list = keys_list[0:33972]
testing_keys_list = keys_list[33972:]

# Training keys
with open("../data/processed/training/FDMO/keys_list.pkl", 'wb') as file_handle:
    pickle.dump(training_keys_list, file_handle)

# Testing keys
with open("../data/processed/training/FDMO/t_keys_list.pkl", 'wb') as file_handle:
    pickle.dump(testing_keys_list, file_handle)

# Split the training and testing datasets
training_seq_msa = seq_msa[0:33972,:]
testing_seq_msa = seq_msa[33972:,:]

# Reweight sequences
# note: only reweighted sequences are used for training.
# seq_msa.shape[0]: number of sequences
# seq_msa.shape[1]: sequence length
seq_weight = np.zeros(training_seq_msa.shape)
for j in range(training_seq_msa.shape[1]):
    aa_type, aa_counts = np.unique(training_seq_msa[:,j], return_counts = True)
    num_type = len(aa_type)
    aa_dict = {}
    for a in aa_type:
        aa_dict[a] = aa_counts[list(aa_type).index(a)]
    for i in range(training_seq_msa.shape[0]):
        seq_weight[i,j] = (1.0/num_type) * (1.0/aa_dict[seq_msa[i,j]])
tot_weight = np.sum(seq_weight)
seq_weight = seq_weight.sum(1) / tot_weight 
with open("../data/processed/training/FDMO/seq_weight.pkl", 'wb') as file_handle:
    pickle.dump(seq_weight, file_handle)

# Testing sequence weight
t_seq_weight = np.ones(testing_seq_msa.shape[0]) / testing_seq_msa.shape[0]
t_seq_weight = t_seq_weight.astype(np.float32)
with open("../data/processed/training/FDMO/t_seq_weight.pkl", 'wb') as file_handle:
    pickle.dump(t_seq_weight, file_handle)

# Remove positions where too many sequences have gaps
pos_idx = []
for i in range(training_seq_msa.shape[1]):
    if np.sum(seq_msa[:,i] == 0) <= seq_msa.shape[0]*0.2:
        pos_idx.append(i)
with open("../data/processed/training/FDMO/seq_pos_idx.pkl", 'wb') as file_handle:
    pickle.dump(pos_idx, file_handle)
    
training_seq_msa = training_seq_msa[:, np.array(pos_idx)]
testing_seq_msa = testing_seq_msa[:, np.array(pos_idx)]
with open("../data/processed/training/FDMO/seq_msa.pkl", 'wb') as file_handle:
    pickle.dump(training_seq_msa, file_handle)
with open("../data/processed/training/FDMO/t_seq_msa.pkl", 'wb') as file_handle:
    pickle.dump(testing_seq_msa, file_handle)

# Change aa numbering into binary
K = 21 # num of classes of aa
D = np.identity(K)
training_num_seq = training_seq_msa.shape[0]
testing_num_seq = testing_seq_msa.shape[0]
len_seq_msa = training_seq_msa.shape[1]
training_seq_msa_binary = np.zeros((training_num_seq, len_seq_msa, K))
testing_seq_msa_binary = np.zeros((testing_num_seq, len_seq_msa, K))
for i in range(training_num_seq):
    training_seq_msa_binary[i,:,:] = D[training_seq_msa[i]]
for j in range(testing_num_seq):
    testing_seq_msa_binary[j,:,:] = D[testing_seq_msa[j]]

# Training binary
with open("../data/processed/training/FDMO/seq_msa_binary.pkl", 'wb') as file_handle:
    pickle.dump(training_seq_msa_binary, file_handle)

# Print train_seq_msa_binary shape
print('Training binary shape: ', training_seq_msa_binary.shape)

# Testing binary
with open("../data/processed/training/FDMO/t_seq_msa_binary.pkl", 'wb') as file_handle:
    pickle.dump(testing_seq_msa_binary, file_handle) 

# Print test_seq_msa_binary shape
print('Testing binary shape: ', testing_seq_msa_binary.shape)